In [15]:
!pip install open_clip_torch transformers statsmodels -q

In [16]:
!pip install -U torch torchvision
!pip install -U open_clip_torch

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
import random
import os
import hashlib
import shutil
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
import open_clip
import pandas as pd


In [18]:
# ========================= CONFIG =========================
# Point these at your two dataset roots.
# Each root may either have Training/Testing sub-folders
# OR direct class folders – loader handles both.
DATASET_DIR        = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"   # 7k dataset root
SECOND_DATASET_DIR = "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri"   # 3k dataset root

IMAGE_SIZE    = 224
N_WAY         = 4
SHOTS         = [1, 3, 5]
EPISODES_EVAL = 600        # raised from 200 → more stable estimates
CI_ALPHA      = 0.05       # 95 % confidence intervals
BOOTSTRAP_N   = 1000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Different datasets use different folder-naming conventions for the same
# classes (e.g. "glioma" vs "glioma_tumor"). Map each canonical class name
# to every folder-name variant we might encounter across dataset roots.
CLASS_NAME_ALIASES = {
    'glioma':     ['glioma', 'glioma_tumor'],
    'meningioma': ['meningioma', 'meningioma_tumor'],
    'notumor':    ['notumor', 'no_tumor'],
    'pituitary':  ['pituitary', 'pituitary_tumor'],
}

print(f"Device : {DEVICE}")
print(f"Classes: {CLASS_NAMES}")
print(f"Episodes (eval): {EPISODES_EVAL}  |  CI: {int((1-CI_ALPHA)*100)}%")

# ---------------------- TIER 1 CONFIG ----------------------
# Tier 2 (unfreeze-last-block episodic meta-training) has been removed.
# All backbones now stay frozen at their pretrained weights throughout the
# notebook. The adaptations below (text fusion, preprocessing, attention
# head) all operate on top of frozen features only, so what gets reported
# is true few-shot generalization -- consistent with the thesis framing
# that the frozen-backbone results are the primary evaluation.

# Tier 1, item 1: text-image fusion (BioMedCLIP only, since it's the only
# backbone with a paired text tower).
ENABLE_TEXT_FUSION = True
TEXT_FUSION_ALPHA  = 0.5     # 1.0 = pure image ProtoNet, 0.0 = pure text zero-shot

# Tier 1, item 2+3: per-image intensity normalization + margin crop
# (cheap skull-strip approximation). margin_frac is fraction cropped off
# EACH side (top/bottom/left/right), so 0.06 removes ~12% of width/height.
INTENSITY_NORMALIZE = True
CROP_MARGIN_FRAC     = 0.06

# Tier 1, item 4: lightweight attention head fit per-episode on frozen features
ATTENTION_STEPS      = 60
ATTENTION_LR         = 0.01
ATTENTION_BOTTLENECK = 64


Device : cuda
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Episodes (eval): 600  |  CI: 95%


In [19]:
# ========================= LOAD CLASS-WISE DATA =========================
def load_dataset_classwise(dataset_root):
    """
    Load image paths from a dataset root that is EITHER:
      • dataset_root/Training/<cls>/ + dataset_root/Testing/<cls>/
      • dataset_root/<cls>/
    Handles class-folder name variants via CLASS_NAME_ALIASES
    (e.g. "glioma" vs "glioma_tumor", "notumor" vs "no_tumor").
    Returns dict  {class_idx: [path, ...]}
    """
    classwise = {i: [] for i in range(N_WAY)}
    train_path = os.path.join(dataset_root, "Training")
    test_path  = os.path.join(dataset_root, "Testing")

    def load_split(split_path):
        if not os.path.exists(split_path):
            return
        for i, cls in enumerate(CLASS_NAMES):
            for alias in CLASS_NAME_ALIASES[cls]:
                cls_dir = os.path.join(split_path, alias)
                if os.path.exists(cls_dir):
                    classwise[i].extend([
                        os.path.join(cls_dir, f)
                        for f in os.listdir(cls_dir)
                    ])
                    break  # matched this class in this split, stop trying aliases

    if os.path.exists(train_path) or os.path.exists(test_path):
        load_split(train_path)
        load_split(test_path)
    else:
        for i, cls in enumerate(CLASS_NAMES):
            for alias in CLASS_NAME_ALIASES[cls]:
                cls_dir = os.path.join(dataset_root, alias)
                if os.path.exists(cls_dir):
                    classwise[i].extend([
                        os.path.join(cls_dir, f)
                        for f in os.listdir(cls_dir)
                    ])
                    break

    return classwise


print("Loading datasets …")
classwise_7k = load_dataset_classwise(DATASET_DIR)
classwise_3k = load_dataset_classwise(SECOND_DATASET_DIR)

# Sanity check: confirm both sources actually contributed images
print("From 7k source:", [len(classwise_7k[i]) for i in range(N_WAY)])
print("From 3k source:", [len(classwise_3k[i]) for i in range(N_WAY)])

# Raw merge
merged_classwise = {i: classwise_7k[i] + classwise_3k[i] for i in range(N_WAY)}

print("Raw counts per class:", [len(merged_classwise[i]) for i in range(N_WAY)])
print("Total raw images   :", sum(len(merged_classwise[i]) for i in range(N_WAY)))

Loading datasets …
From 7k source: [1800, 1800, 1800, 1800]
From 3k source: [926, 937, 500, 901]
Raw counts per class: [2726, 2737, 2300, 2701]
Total raw images   : 10464


In [20]:
# ========================= DUPLICATE REMOVAL =========================
def hash_image(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def remove_duplicates_classwise(classwise):
    """Remove exact-duplicate images (same MD5) within each class."""
    cleaned, total_removed = {}, 0
    for cls, paths in classwise.items():
        seen, new_list = set(), []
        for p in paths:
            h = hash_image(p)
            if h not in seen:
                seen.add(h)
                new_list.append(p)
            else:
                total_removed += 1
        cleaned[cls] = new_list
    return cleaned, total_removed


print("Removing intra-dataset duplicates …")
merged_classwise, n_removed = remove_duplicates_classwise(merged_classwise)
print(f"Removed {n_removed} duplicate image(s).")
print("Clean counts per class:", [len(merged_classwise[i]) for i in range(N_WAY)])


Removing intra-dataset duplicates …
Removed 2879 duplicate image(s).
Clean counts per class: [2301, 1819, 1689, 1776]


In [21]:
# ========================= LEAK-FREE TRAIN / TEST SPLIT =========================
def split_classwise_safe(classwise, train_ratio=0.8, seed=SEED):
    """
    Deterministic, per-class stratified split.
    Uses a local RNG so it never disturbs the global state.
    Returns support_pool (train) and query_pool (test) dicts.
    """
    rng = random.Random(seed)
    support, query = {}, {}
    for cls, imgs in classwise.items():
        imgs_copy = list(imgs)
        rng.shuffle(imgs_copy)
        split        = int(len(imgs_copy) * train_ratio)
        support[cls] = imgs_copy[:split]
        query[cls]   = imgs_copy[split:]
    return support, query


support_pool, query_pool = split_classwise_safe(merged_classwise, train_ratio=0.8)

# ── Leakage check ──
print("\n================ LEAKAGE CHECK ================")
train_set = set(p for paths in support_pool.values() for p in paths)
test_set  = set(p for paths in query_pool.values()   for p in paths)

path_overlap = train_set & test_set

train_hashes = {hash_image(p) for p in train_set}
test_hashes  = {hash_image(p) for p in test_set}
hash_overlap = train_hashes & test_hashes

print(f"Path overlap        : {len(path_overlap)}")
print(f"Hash (pixel) overlap: {len(hash_overlap)}")

assert len(path_overlap) == 0, "PATH LEAKAGE detected!"
assert len(hash_overlap) == 0, "PIXEL LEAKAGE detected!"
print("✅ PASS – No leakage detected")

print("\n================ SPLIT STATS ================")
print(f"Support (train) per class: {[len(support_pool[i]) for i in range(N_WAY)]}")
print(f"Query   (test)  per class: {[len(query_pool[i])   for i in range(N_WAY)]}")
print(f"Total support: {sum(len(support_pool[i]) for i in range(N_WAY))}")
print(f"Total query  : {sum(len(query_pool[i])   for i in range(N_WAY))}")



================ LEAKAGE CHECK ================
Path overlap        : 0
Hash (pixel) overlap: 0
✅ PASS – No leakage detected

================ SPLIT STATS ================
Support (train) per class: [1840, 1455, 1351, 1420]
Query   (test)  per class: [461, 364, 338, 356]
Total support: 6066
Total query  : 1519


In [22]:
# ========================= EPISODE CREATION =========================
def create_episode(k_shot, support_classwise, query_classwise, query_per_class=15):
    """
    Samples support from support_pool and query from query_pool ONLY.
    No cross-contamination is possible because the two pools are disjoint.
    """
    support_paths, query_paths   = [], []
    support_labels, query_labels = [], []

    for cls_idx in range(N_WAY):
        s_imgs = support_classwise[cls_idx]
        s_sel  = (random.choices(s_imgs, k=k_shot)
                  if len(s_imgs) < k_shot
                  else random.sample(s_imgs, k_shot))

        q_imgs = query_classwise[cls_idx]
        q_sel  = (random.choices(q_imgs, k=query_per_class)
                  if len(q_imgs) < query_per_class
                  else random.sample(q_imgs, query_per_class))

        support_paths.extend(s_sel)
        query_paths.extend(q_sel)
        support_labels.extend([cls_idx] * k_shot)
        query_labels.extend([cls_idx] * query_per_class)

    return (
        support_paths,
        query_paths,
        torch.tensor(support_labels, dtype=torch.long),
        torch.tensor(query_labels,   dtype=torch.long),
    )


In [23]:
# ========================= MODELS =========================
print("Loading BioMedCLIP …")
biomed_model, biomed_preprocess = open_clip.create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
biomed_model = biomed_model.to(DEVICE)     # full model (visual + text tower) — needed for Tier 1 text fusion
biomed_model.eval()
biomed_visual = biomed_model.visual        # kept for backward-compat with the rest of the notebook
biomed_visual.eval()

print("Loading ResNet18 …")
resnet    = models.resnet18(pretrained=True)
resnet.fc = nn.Identity()          # 512-dim output
resnet    = resnet.to(DEVICE)
resnet.eval()

print("Loading EfficientNet-B0 …")
efficientnet             = models.efficientnet_b0(pretrained=True)
efficientnet.classifier  = nn.Identity()   # 1280-dim output – kept as-is
efficientnet             = efficientnet.to(DEVICE)
efficientnet.eval()

imagenet_preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

model_configs = {
    "BioMedCLIP"     : {"encoder": biomed_visual,  "preprocess": biomed_preprocess, "dim": 768},
    "ResNet18"        : {"encoder": resnet,          "preprocess": imagenet_preprocess, "dim": 512},
    "EfficientNet-B0" : {"encoder": efficientnet,    "preprocess": imagenet_preprocess, "dim": 1280},
}

print("✅ All models loaded.")


Loading BioMedCLIP …
Loading ResNet18 …
Loading EfficientNet-B0 …
✅ All models loaded.


In [24]:
# ========================= TIER 1: PREPROCESSING UPGRADES =========================
# Items 2 & 3 from the plan: per-image intensity normalization + a cheap
# skull-strip approximation (margin crop). Both operate at the PIL stage,
# BEFORE each model's own resize/normalize pipeline, so they compose
# cleanly with the existing biomed_preprocess / imagenet_preprocess objects.

def intensity_normalize(img: Image.Image) -> Image.Image:
    """
    Per-image min-max intensity normalization.
    MRI intensity ranges vary scanner-to-scanner and protocol-to-protocol;
    rescaling each image to its own full [0, 255] range makes contrast more
    consistent across the merged 7k+3k sources than relying only on the
    fixed ImageNet mean/std applied later.
    """
    arr = np.asarray(img.convert("RGB")).astype(np.float32)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-6:
        return img  # flat image — nothing to normalize
    arr = (arr - lo) / (hi - lo) * 255.0
    return Image.fromarray(arr.astype(np.uint8))


def center_crop_margin(img: Image.Image, margin_frac: float = CROP_MARGIN_FRAC) -> Image.Image:
    """
    Cheap skull-strip approximation: crop a fixed margin off each side to
    remove scanner border / background and nudge out some skull edge,
    without needing a full FSL BET-style segmentation pipeline.
    """
    w, h = img.size
    dw, dh = int(w * margin_frac), int(h * margin_frac)
    if dw == 0 or dh == 0:
        return img
    return img.crop((dw, dh, w - dw, h - dh))


def build_tier1_preprocess(base_preprocess):
    """Wrap a model's existing preprocess() with the Tier-1 PIL-level steps."""
    def _combined(img):
        if INTENSITY_NORMALIZE:
            img = intensity_normalize(img)
        if CROP_MARGIN_FRAC > 0:
            img = center_crop_margin(img)
        return base_preprocess(img)
    return _combined


for _name, _cfg in model_configs.items():
    _cfg["preprocess"] = build_tier1_preprocess(_cfg["preprocess"])

print(f"✅ Tier-1 preprocessing wired in for all models "
      f"(intensity_normalize={INTENSITY_NORMALIZE}, crop_margin_frac={CROP_MARGIN_FRAC}).")


✅ Tier-1 preprocessing wired in for all models (intensity_normalize=True, crop_margin_frac=0.06).


In [25]:
# ========================= TIER 1: TEXT PROTOTYPES (BioMedCLIP text tower) =========================
# Item 1 from the plan: you already load the full BiomedCLIP model (with its
# PubMedBERT text tower) in the MODELS cell but only ever use `.visual`. This
# cell encodes a handful of radiology-style prompts per class and averages
# them into one L2-normalized text prototype per class, for later fusion
# with the image prototypes (Tip-Adapter-style) in text_fusion_episode().
#
# NOTE: this only applies to BioMedCLIP — ResNet18 / EfficientNet-B0 have no
# paired text tower, so text_fusion is skipped for them (handled in run_fewshot).

text_prototypes = None

if ENABLE_TEXT_FUSION:
    print("Building BiomedCLIP text prototypes (Tier 1 — text/image fusion) …")

    tokenizer = open_clip.get_tokenizer(
        'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    )

    # A few radiology-style prompt variants per class (prompt-ensembling,
    # same spirit as CLIP zero-shot classification / TumorCLIP).
    CLASS_TEXT_PROMPTS = {
        0: [  # glioma
            "a T1-weighted axial brain MRI showing a glioma tumor",
            "an MRI scan of the brain with a glioma lesion",
            "brain magnetic resonance image showing glial tumor tissue",
        ],
        1: [  # meningioma
            "a T1-weighted axial brain MRI showing a meningioma tumor",
            "an MRI scan of the brain with a meningioma lesion",
            "brain magnetic resonance image showing a meningeal tumor",
        ],
        2: [  # notumor
            "a T1-weighted axial brain MRI with no tumor, healthy brain tissue",
            "a normal brain MRI scan without any tumor",
            "brain magnetic resonance image showing no abnormal mass",
        ],
        3: [  # pituitary
            "a T1-weighted axial brain MRI showing a pituitary tumor",
            "an MRI scan of the brain with a pituitary gland lesion",
            "brain magnetic resonance image showing a pituitary adenoma",
        ],
    }

    try:
        _protos = []
        with torch.no_grad():
            for cls_idx in range(N_WAY):
                prompts   = CLASS_TEXT_PROMPTS[cls_idx]
                tokens    = tokenizer(prompts).to(DEVICE)
                txt_feat  = biomed_model.encode_text(tokens)                 # [n_prompts, dim]
                txt_feat  = F.normalize(txt_feat, p=2, dim=-1)
                proto     = F.normalize(txt_feat.mean(dim=0), p=2, dim=0)    # ensemble prompts
                _protos.append(proto.cpu())
        text_prototypes = torch.stack(_protos)   # [N_WAY, text_dim]

        img_dim = model_configs["BioMedCLIP"]["dim"]
        print(f"✅ Text prototypes built: shape {tuple(text_prototypes.shape)}")
        if text_prototypes.shape[-1] != img_dim:
            print(f"  ⚠ WARNING: text dim ({text_prototypes.shape[-1]}) != configured BioMedCLIP "
                  f"image dim ({img_dim}). text_fusion will auto-skip at eval time if this "
                  f"mismatch is real — re-check model_configs['BioMedCLIP']['dim'] against the "
                  f"actual cached feature shape from the caching cell below.")
    except Exception as e:
        print(f"  ⚠ Could not build text prototypes ({e}). Disabling text_fusion for this run.")
        text_prototypes = None
else:
    print("Text fusion disabled (ENABLE_TEXT_FUSION=False).")


Building BiomedCLIP text prototypes (Tier 1 — text/image fusion) …
✅ Text prototypes built: shape (4, 512)
  ⚠ WARNING: text dim (512) != configured BioMedCLIP image dim (768). text_fusion will auto-skip at eval time if this mismatch is real — re-check model_configs['BioMedCLIP']['dim'] against the actual cached feature shape from the caching cell below.


In [ ]:
# ========================= FEATURE CACHING =========================
# Pre-extract all features once → evaluation is fast (no per-episode GPU calls).
# Cache dir renamed (v2) so this always re-extracts fresh — preprocessing
# just changed (Tier 1) and backbones stay frozen at pretrained weights,
# so any stale "feature_cache_merged" from a previous run must NOT be reused.
CACHE_DIR = "feature_cache_merged_v2"
if os.path.exists(CACHE_DIR):
    print("Clearing stale cache …")
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)


def extract_features(name, encoder, preprocess, classwise_dict, tag):
    cache_file = os.path.join(CACHE_DIR, f"{name}_{tag}.pt")
    if os.path.exists(cache_file):
        print(f"  ✓ Loaded cache [{tag}] {name}")
        return torch.load(cache_file, map_location="cpu")

    all_paths = [p for paths in classwise_dict.values() for p in paths]
    features  = {}

    with torch.no_grad():
        for path in tqdm(all_paths, desc=f"{name}[{tag}]"):
            try:
                img    = Image.open(path).convert("RGB")
                tensor = preprocess(img).unsqueeze(0).to(DEVICE)
                feat   = encoder(tensor).squeeze(0).cpu()
                features[path] = feat
            except Exception as e:
                print(f"  ⚠ Skipped {path}: {e}")

    torch.save(features, cache_file)
    return features


print("Pre-extracting features (support pool) …")
support_feats = {}
query_feats   = {}

for name, cfg in model_configs.items():
    support_feats[name] = extract_features(
        name, cfg["encoder"], cfg["preprocess"], support_pool, "support"
    )
    query_feats[name] = extract_features(
        name, cfg["encoder"], cfg["preprocess"], query_pool, "query"
    )

print("\n✅ Feature extraction complete.")

# Sanity check for Tier 1 text fusion: confirm image/text embedding dims actually match.
if ENABLE_TEXT_FUSION and text_prototypes is not None:
    _sample_feat = next(iter(support_feats["BioMedCLIP"].values()))
    if _sample_feat.shape[-1] != text_prototypes.shape[-1]:
        print(f"  ⚠ BioMedCLIP cached feature dim ({_sample_feat.shape[-1]}) != text prototype dim "
              f"({text_prototypes.shape[-1]}). text_fusion will be skipped automatically at eval time.")
    else:
        print(f"  ✅ BioMedCLIP image/text dims match ({_sample_feat.shape[-1]}) — text_fusion is usable.")


Clearing stale cache …
Pre-extracting features (support pool) …


ResNet18[support]:  89%|████████▉ | 5387/6066 [01:00<00:08, 83.03it/s] 

In [ ]:
# ========================= 95% CONFIDENCE INTERVAL =========================
def bootstrap_ci(acc_list, n=BOOTSTRAP_N, alpha=CI_ALPHA):
    arr     = np.array(acc_list)
    samples = np.random.choice(arr, size=(n, len(arr)), replace=True).mean(axis=1)
    lo      = np.percentile(samples, 100 * alpha / 2)
    hi      = np.percentile(samples, 100 * (1 - alpha / 2))
    return lo, hi


In [ ]:
# ========================= VISUALIZATION HELPERS =========================
SAVE_DIR = "output_merged"
os.makedirs(SAVE_DIR, exist_ok=True)

_BAR_COLORS    = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#795548']
_LINE_COLORS   = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A', '#4E342E']
_LINE_MARKERS  = ['o', 's', '^', 'D', 'v']


def save_fig(fig, filename, dpi=200):
    path = os.path.join(SAVE_DIR, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Saved: {path}")
    return path


def plot_confusion_matrix(cm, name, k_shot, mean_acc, std_acc, ci_lo, ci_hi, method="prototypical"):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor='gray', ax=ax
    )
    ax.set_title(
        f'{name} – {k_shot}-shot ({method}) (Merged Dataset)\n'
        f'Acc: {mean_acc:.3f} ± {std_acc:.3f}  |  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]',
        fontsize=12, fontweight='bold', pad=12
    )
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.tick_params(axis='x', rotation=30)
    save_fig(fig, f"cm_{name.replace('-','_')}_{k_shot}shot_{method}_merged.png")


# NOTE: method= was already being passed in by the RUN ALL EXPERIMENTS /
# SUMMARY VISUALIZATIONS cells below but wasn't accepted here, and
# plot_summary_bar referenced an undefined `method` variable inside its own
# body — both fixed here (now a proper parameter used consistently).
def plot_summary_bar(results, shots, model_names, method="prototypical"):
    x      = np.arange(len(model_names))
    width  = 0.22
    colors = (_BAR_COLORS * 3)[:len(model_names)]
    fig, axes = plt.subplots(1, len(shots), figsize=(6 * len(shots), 5), sharey=True)
    if len(shots) == 1:
        axes = [axes]
    for ax, k in zip(axes, shots):
        accs = [results[f"{m}_{k}shot"]['mean_acc'] for m in model_names]
        f1s  = [results[f"{m}_{k}shot"]['macro_f1'] for m in model_names]
        cis  = np.array([
            (results[f"{m}_{k}shot"]['mean_acc'] - results[f"{m}_{k}shot"]['ci_lo'],
             results[f"{m}_{k}shot"]['ci_hi'] - results[f"{m}_{k}shot"]['mean_acc'])
            for m in model_names
        ]).T
        ax.bar(x - width/2, accs, width, color=colors, alpha=0.88,
               yerr=cis, capsize=5, error_kw={'elinewidth': 1.5}, label='Accuracy')
        ax.bar(x + width/2, f1s,  width, color=colors, alpha=0.45,
               hatch='//', label='Macro-F1')
        for i, (a, f) in enumerate(zip(accs, f1s)):
            ax.text(i - width/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=8)
        ax.set_title(f'{k}-Shot', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(model_names, rotation=15, ha='right')
        ax.set_ylim(0, 1.08)
        ax.set_ylabel('Score')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
    fig.suptitle(f'Few-Shot Brain Tumor MRI Classification (Merged 7k+3k)\n'
                 f'Leak-Free | {method} | 95% CI', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, f"summary_acc_f1_{method}_merged.png")


def plot_shot_progression(results, shots, model_names, method="prototypical"):
    markers = (_LINE_MARKERS * 3)[:len(model_names)]
    colors  = (_LINE_COLORS * 3)[:len(model_names)]
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, name in enumerate(model_names):
        accs  = [results[f"{name}_{k}shot"]['mean_acc'] for k in shots]
        ci_lo = [results[f"{name}_{k}shot"]['ci_lo']    for k in shots]
        ci_hi = [results[f"{name}_{k}shot"]['ci_hi']    for k in shots]
        ax.plot(shots, accs, marker=markers[i], color=colors[i],
                linewidth=2.2, markersize=8, label=name)
        ax.fill_between(shots, ci_lo, ci_hi, color=colors[i], alpha=0.12)
    ax.axhline(1 / N_WAY, color='gray', linestyle='--',
               linewidth=1.5, label=f'Random ({1/N_WAY:.2f})')
    ax.set_xlabel('Shots per class (k)')
    ax.set_ylabel('Mean Accuracy')
    ax.set_title(f'Accuracy vs. k-shot ({method}, shaded = 95% CI) – Merged Dataset',
                 fontsize=12, fontweight='bold')
    ax.set_xticks(shots)
    ax.set_xticklabels([f'{k}-shot' for k in shots])
    ax.set_ylim(0.1, 1.05)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    save_fig(fig, f"shot_progression_{method}_merged.png")


def plot_perclass_heatmap(results, shots, model_names, method="prototypical"):
    row_labels = [f"{m}\n({k}-shot)" for k in shots for m in model_names]
    data = np.array([
        results[f"{m}_{k}shot"]['per_class_f1']
        for k in shots for m in model_names
    ])
    fig, ax = plt.subplots(figsize=(8, max(5, len(row_labels) * 0.55 + 1.5)))
    sns.heatmap(data, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=CLASS_NAMES, yticklabels=row_labels,
                vmin=0, vmax=1, linewidths=0.4, linecolor='white', ax=ax)
    ax.set_title(f'Per-Class F1 Score ({method}, all models × shots) – Merged Dataset',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Model / Shot')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    save_fig(fig, f"perclass_f1_heatmap_{method}_merged.png")


In [ ]:
# ========================= ADD THIS NEW FUNCTION HERE =========================
def filter_by_method(results, method):
    """Strip the trailing _method suffix so old plotting functions still work."""
    suffix = f"_{method}"
    return {k[: -len(suffix)]: v for k, v in results.items() if k.endswith(suffix)}


In [ ]:
# ========================= FINE-TUNED CLASSIFIER =========================
class CosineClassifier(nn.Module):
    """Learnable cosine-similarity classifier — Chen et al. 'Baseline++' (ICLR 2019)."""
    def __init__(self, feat_dim, n_way):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_way, feat_dim) * 0.01)
        self.temperature = nn.Parameter(torch.tensor(10.0))

    def forward(self, x):
        x_n = F.normalize(x, dim=-1)
        w_n = F.normalize(self.weight, dim=-1)
        return self.temperature * torch.mm(x_n, w_n.t())


def finetune_episode(sf, sy, qf, n_way=N_WAY, steps=50, lr=0.01):
    """Fine-tune a fresh classifier head on ONE episode's support set only."""
    feat_dim = sf.shape[1]
    clf = CosineClassifier(feat_dim, n_way).to(DEVICE)
    optimizer = torch.optim.Adam(clf.parameters(), lr=lr)

    clf.train()
    for _ in range(steps):
        optimizer.zero_grad()
        logits = clf(sf)
        loss = F.cross_entropy(logits, sy)
        loss.backward()
        optimizer.step()

    clf.eval()
    with torch.no_grad():
        preds = clf(qf).argmax(dim=1)
    return preds


# ========================= TIER 1, ITEM 4: ATTENTION HEAD ON FROZEN FEATURES =========================
class FeatureAttentionHead(nn.Module):
    """
    Lightweight channel-attention head applied on top of FROZEN cached
    features. A squeeze-excitation-style gate plus a residual linear
    re-weighting — cheap enough to fit fresh per episode, no backbone
    gradients required. Loosely follows Ouahab et al.'s attention-augmented
    ProtoNet (92.44% on a 3-class set), applied here on top of your cache.
    """
    def __init__(self, feat_dim, bottleneck=ATTENTION_BOTTLENECK):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(feat_dim, bottleneck),
            nn.ReLU(inplace=True),
            nn.Linear(bottleneck, feat_dim),
            nn.Sigmoid(),
        )
        self.proj = nn.Linear(feat_dim, feat_dim)

    def forward(self, x):
        g = self.gate(x)
        return F.normalize(self.proj(x) * g + x, p=2, dim=-1)  # residual + renormalize


def attention_episode(sf, sy, qf, n_way=N_WAY, steps=ATTENTION_STEPS, lr=ATTENTION_LR):
    """
    Fits a fresh FeatureAttentionHead on ONE episode's support set (via a
    prototypical loss), then classifies the query set through the same
    attended feature space. Backbone stays frozen — this only re-weights
    the cached vectors.
    """
    feat_dim = sf.shape[1]
    head = FeatureAttentionHead(feat_dim).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)

    head.train()
    for _ in range(steps):
        optimizer.zero_grad()
        sf_att = head(sf)
        prototypes = torch.stack([sf_att[sy == i].mean(dim=0) for i in range(n_way)])
        dists = 1 - torch.mm(sf_att, prototypes.t())
        loss = F.cross_entropy(-dists, sy)
        loss.backward()
        optimizer.step()

    head.eval()
    with torch.no_grad():
        sf_att = head(sf)
        qf_att = head(qf)
        prototypes = torch.stack([sf_att[sy == i].mean(dim=0) for i in range(n_way)])
        dists = 1 - torch.mm(qf_att, prototypes.t())
        preds = dists.argmin(dim=1)
    return preds


# ========================= TIER 1, ITEM 1: TEXT-IMAGE FUSION (BioMedCLIP only) =========================
def text_fusion_episode(sf, sy, qf, text_prototypes, n_way=N_WAY, alpha=TEXT_FUSION_ALPHA):
    """
    Tip-Adapter-style fusion: blend the usual image-only prototypes with
    BiomedCLIP's frozen text-derived class prototypes.
      alpha = 1.0  ->  pure image ProtoNet (same as 'prototypical' method)
      alpha = 0.0  ->  pure text zero-shot classification
    """
    device = sf.device
    text_prototypes = text_prototypes.to(device)
    if text_prototypes.shape[-1] != sf.shape[-1]:
        raise ValueError(
            f"text_fusion dim mismatch: image feat dim {sf.shape[-1]} vs "
            f"text proto dim {text_prototypes.shape[-1]}."
        )

    image_prototypes = torch.stack([sf[sy == i].mean(dim=0) for i in range(n_way)])
    image_prototypes = F.normalize(image_prototypes, p=2, dim=-1)
    text_p = F.normalize(text_prototypes, p=2, dim=-1)

    fused_prototypes = F.normalize(alpha * image_prototypes + (1 - alpha) * text_p, p=2, dim=-1)
    dists = 1 - torch.mm(qf, fused_prototypes.t())
    preds = dists.argmin(dim=1)
    return preds


In [ ]:
# ========================= EVALUATION (Leak-Free) =========================
def run_fewshot(k_shot, name, method="prototypical"):
    # text_fusion only makes sense for BioMedCLIP (only backbone with a paired
    # text tower) and only when the text prototypes actually built successfully.
    if method == "text_fusion" and (name != "BioMedCLIP" or text_prototypes is None):
        print(f"  ⏭  Skipping text_fusion for {name} (no text prototypes available).")
        return None

    print(f"\n{'='*80}")
    print(f"  {k_shot}-SHOT | {name} | {method}  (Merged 7k+3k | Leak-Free)")
    print(f"{'='*80}")

    s_feats = support_feats[name]
    q_feats = query_feats[name]

    all_true, all_pred = [], []
    episode_preds      = []
    accs               = []

    for _ in tqdm(range(EPISODES_EVAL), desc=f"{k_shot}-shot {name} [{method}]"):
        supp_paths, qry_paths, sy, qy = create_episode(
            k_shot, support_pool, query_pool, query_per_class=15
        )
        sy, qy = sy.to(DEVICE), qy.to(DEVICE)

        sf = torch.stack([s_feats[p] for p in supp_paths]).to(DEVICE)
        qf = torch.stack([q_feats[p] for p in qry_paths ]).to(DEVICE)

        # L2-normalise → cosine similarity via dot-product
        sf = F.normalize(sf, p=2, dim=-1)
        qf = F.normalize(qf, p=2, dim=-1)

        # ===== METHOD DISPATCH =====
        if method == "prototypical":
            prototypes = torch.stack([sf[sy == i].mean(dim=0) for i in range(N_WAY)])
            dists = 1 - torch.mm(qf, prototypes.t())
            preds = dists.argmin(dim=1)
        elif method == "finetuned":
            preds = finetune_episode(sf, sy, qf)
        elif method == "attention":
            preds = attention_episode(sf, sy, qf)
        elif method == "text_fusion":
            preds = text_fusion_episode(sf, sy, qf, text_prototypes)
        else:
            raise ValueError(f"Unknown method: {method}")
        # ============================

        ep_true = qy.cpu().numpy()
        ep_pred = preds.cpu().numpy()

        all_true.extend(ep_true)
        all_pred.extend(ep_pred)
        episode_preds.append(ep_pred)
        accs.append(accuracy_score(ep_true, ep_pred))

    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    ci_lo, ci_hi = bootstrap_ci(accs)

    per_p, per_r, per_f1, support_counts = precision_recall_fscore_support(
        all_true, all_pred, average=None, zero_division=0, labels=list(range(N_WAY))
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        all_true, all_pred, average='macro', zero_division=0
    )
    cm = confusion_matrix(all_true, all_pred)

    print(f"\n  Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"  95% CI   : [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  Macro F1 : {macro_f1:.4f}  |  Precision: {macro_p:.4f}  |  Recall: {macro_r:.4f}")
    print("\n  Per-class metrics:")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"    {cls:12s}  P:{per_p[i]:.3f}  R:{per_r[i]:.3f}  F1:{per_f1[i]:.3f}  N:{support_counts[i]}")

    plot_confusion_matrix(cm, name, k_shot, mean_acc, std_acc, ci_lo, ci_hi, method=method)

    return {
        'mean_acc'    : mean_acc,
        'std_acc'     : std_acc,
        'ci_lo'       : ci_lo,
        'ci_hi'       : ci_hi,
        'macro_f1'    : macro_f1,
        'macro_p'     : macro_p,
        'macro_r'     : macro_r,
        'per_class_f1': per_f1,
        'all_true'    : all_true,
        'all_pred'    : all_pred,
        'episode_preds': episode_preds,
    }


In [ ]:
# ========================= RUN ALL EXPERIMENTS =========================
results      = {}
model_names  = list(model_configs.keys())
methods      = ["prototypical", "finetuned", "attention", "text_fusion"]  # Tier 1 additions: attention, text_fusion

print(f"\n{'#'*80}")
print("FEW-SHOT EXPERIMENTS ON MERGED DATASET (7k + 3k)")
print(f"{'#'*80}\n")

for k in SHOTS:
    for name in model_names:
        for method in methods:
            key = f"{name}_{k}shot_{method}"
            r = run_fewshot(k_shot=k, name=name, method=method)
            if r is not None:      # text_fusion returns None for non-BioMedCLIP models
                results[key] = r

print("\n✅ All experiments complete.")


In [ ]:
# ========================= STATISTICAL SIGNIFICANCE (McNemar's Test) =========================
print(f"\n{'#'*80}")
print("MCNEMAR'S TEST")
print(f"{'#'*80}")

def mcnemar_compare(pred_a, pred_b, true, label_a, label_b):
    ya, yb, yt = np.array(pred_a), np.array(pred_b), np.array(true)
    aa = np.sum((ya == yt) & (yb == yt))
    bb = np.sum((ya == yt) & (yb != yt))
    cc = np.sum((ya != yt) & (yb == yt))
    dd = np.sum((ya != yt) & (yb != yt))
    table = [[aa, bb], [cc, dd]]
    try:
        test = mcnemar(table, exact=False, correction=True)
        sig = "significant" if test.pvalue < CI_ALPHA else "n.s."
        print(f"  {label_a} vs {label_b}: chi2={test.statistic:.3f}  p={test.pvalue:.4f}  ({sig})")
    except Exception as e:
        print(f"  {label_a} vs {label_b}: could not compute ({e})")

# (A) Model vs model, within each method
# (guarded: text_fusion only has BioMedCLIP results, so a missing key just gets skipped)
for method in methods:
    for k in SHOTS:
        print(f"\n--- {k}-Shot | {method} | model comparisons ---")
        for i in range(len(model_names)):
            for j in range(i + 1, len(model_names)):
                ma, mb = model_names[i], model_names[j]
                key_a, key_b = f"{ma}_{k}shot_{method}", f"{mb}_{k}shot_{method}"
                if key_a not in results or key_b not in results:
                    continue
                ra, rb = results[key_a], results[key_b]
                mcnemar_compare(ra['all_pred'], rb['all_pred'], ra['all_true'], ma, mb)

# (B) Pairwise method comparisons, within each model (generalized to cover all
# 4 methods now, not just prototypical vs finetuned — still guarded against
# the sparse text_fusion keys).
print(f"\n--- pairwise method comparisons (per model, per shot) ---")
for k in SHOTS:
    for name in model_names:
        available = [m for m in methods if f"{name}_{k}shot_{m}" in results]
        for i in range(len(available)):
            for j in range(i + 1, len(available)):
                ma, mb = available[i], available[j]
                ra = results[f"{name}_{k}shot_{ma}"]
                rb = results[f"{name}_{k}shot_{mb}"]
                mcnemar_compare(ra['all_pred'], rb['all_pred'], ra['all_true'],
                                 f"{name} {ma}", f"{name} {mb}")


In [ ]:
# ========================= FINAL RESULTS TABLE =========================
print(f"\n{'='*100}")
print("FINAL SUMMARY – Merged 7k+3k | Leak-Free")
print(f"{'='*100}")
print(f"{'Shot':<6} {'Model':<18} {'Method':<13} {'Acc':>8} {'±Std':>7} {'CI-Lo':>7} {'CI-Hi':>7} "
      f"{'F1':>7} {'Prec':>7} {'Rec':>7}")
print("-" * 100)

rows = []
for k in SHOTS:
    for name in model_names:
        for method in methods:
            key = f"{name}_{k}shot_{method}"
            if key not in results:      # e.g. text_fusion for non-BioMedCLIP models
                continue
            r = results[key]
            print(f"{k:<6} {name:<18} {method:<13} {r['mean_acc']:>8.4f} {r['std_acc']:>7.4f} "
                  f"{r['ci_lo']:>7.4f} {r['ci_hi']:>7.4f} "
                  f"{r['macro_f1']:>7.4f} {r['macro_p']:>7.4f} {r['macro_r']:>7.4f}")
            rows.append({
                'Shot': k, 'Model': name, 'Method': method,
                'Accuracy': r['mean_acc'], 'Std': r['std_acc'],
                'CI_Lo': r['ci_lo'], 'CI_Hi': r['ci_hi'],
                'F1': r['macro_f1'], 'Precision': r['macro_p'], 'Recall': r['macro_r'],
            })
    print("-" * 100)

csv_path = os.path.join(SAVE_DIR, "results_merged.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"\n✅ Results saved to {csv_path}")


In [ ]:
# ========================= SUMMARY VISUALIZATIONS =========================
print("\nGenerating summary figures …")
for method in methods:
    filtered = filter_by_method(results, method)
    if not filtered:
        print(f"  ⏭  No results for method='{method}', skipping its figures.")
        continue

    # text_fusion only has BioMedCLIP entries — derive which models actually
    # have results for this method instead of assuming all of model_names.
    models_present = sorted(
        {k.split('_', 1)[0] for k in filtered.keys()},
        key=lambda m: model_names.index(m)
    )

    plot_summary_bar(filtered, SHOTS, models_present, method=method)
    plot_shot_progression(filtered, SHOTS, models_present, method=method)
    plot_perclass_heatmap(filtered, SHOTS, models_present, method=method)

print("\n" + "="*60)
print("All outputs written to:", SAVE_DIR)
print("="*60)
